# nb_lakehouse_health_audit — table inventory & health report
**Platform pattern:** observability before action. This notebook only *reads* — it walks every Delta
table under a Lakehouse root, gathers health signals from Delta's own APIs (`DESCRIBE DETAIL`,
`SHOW TBLPROPERTIES`, `DESCRIBE HISTORY`), and writes a versioned **health report table** that the
maintenance notebook (and humans) act on. Runs unchanged locally or in Fabric
(`TABLES_ROOT = "/lakehouse/default/Tables"`).

In [1]:
# PARAMETERS (Fabric: toggle "parameters" on this cell; pipeline/runMultiple can override)
NOTEBOOK_NAME = "nb_lakehouse_health_audit"
TABLES_ROOT   = "/tmp/fabric_kit_warehouse"      # Fabric: /lakehouse/default/Tables
REPORT_TABLE  = f"{TABLES_ROOT}/_ops/table_health"
SMALL_FILE_MB = 64          # avg file size below this (with MIN_FILES+) flags compaction
MIN_FILES     = 8

In [2]:
# --- Session: Fabric is the default target -------------------------------------
# In Fabric you do NOT create a Spark session. The Livy layer starts it before your first
# cell runs, and `spark` (plus `sc`, `notebookutils`) are already bound. Calling
# SparkSession.builder there is at best a no-op via getOrCreate() and at worst misleading:
# master(), Delta wiring and executor shape are all decided by the Environment/pool, not here.
#
# Session-start settings belong in a %%configure -f cell ABOVE this one, or in the
# Environment's Spark properties. Only runtime-mutable keys can be set from code.
try:
    spark                      # noqa: F821  <- Fabric (and any live session): already provided
    IN_FABRIC = True
except NameError:
    # Local/dev fallback ONLY. Never runs in Fabric.
    IN_FABRIC = False
    from pyspark.sql import SparkSession
    from delta import configure_spark_with_delta_pip
    _b = (SparkSession.builder.appName(NOTEBOOK_NAME).master("local[4]")
          .config("spark.driver.memory", "2g")
          .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
          .config("spark.sql.catalog.spark_catalog",
                  "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
    spark = configure_spark_with_delta_pip(_b).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print(("Fabric session (provided)" if IN_FABRIC else "local session (dev fallback)"),
      "| Spark", spark.version)

# --- session bootstrap (identical in every kit notebook; Fabric supplies `spark`) ---
import os, sys, json, time
from datetime import datetime, timezone

def get_session():
    try:
        return spark  # noqa: F821  (Fabric / existing session)
    except NameError:
        from pyspark.sql import SparkSession
        from delta import configure_spark_with_delta_pip
        b = (SparkSession.builder.appName(NOTEBOOK_NAME).master("local[4]")
             .config("spark.driver.memory", "2g")
             .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
             .config("spark.sql.catalog.spark_catalog",
                     "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
        return configure_spark_with_delta_pip(b).getOrCreate()

spark = get_session()
spark.sparkContext.setLogLevel("ERROR")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
APP_ID = spark.sparkContext.applicationId
print(f"{NOTEBOOK_NAME} | run {RUN_ID} | app {APP_ID} | Spark {spark.version}")

26/08/04 12:07:25 WARN Utils: Your hostname, vm resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/08/04 12:07:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/usr/local/lib/python3.12/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1e334dce-0a06-494e-8b4a-1cace7396627;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central


	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 264ms :: artifacts dl 21ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0   ||   3   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-1e334dce-0a06-494e-8b4a-1cace7396627
	confs: [default]
	0 artifacts copied, 3 already retrieved (0kB/8ms)


26/08/04 12:07:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


local session (dev fallback) | Spark 3.5.1
nb_lakehouse_health_audit | run 20260804T120729Z | app local-1785845248131 | Spark 3.5.1


In [3]:
# --- self-contained demo data (skipped if the root already has tables) ---
from pyspark.sql import functions as F
import os
os.makedirs(TABLES_ROOT, exist_ok=True)
def _has_delta(root):
    return any(os.path.isdir(os.path.join(root, d, "_delta_log")) for d in os.listdir(root)) if os.path.isdir(root) else False
if not _has_delta(TABLES_ROOT):
    (spark.range(0, 100_000)
        .withColumn("customer_id", (F.col("id") % 500).cast("int"))
        .withColumn("amount", F.round(F.rand() * 500, 2))
        .withColumn("status", F.when(F.col("id") % 7 == 0, "cancelled").otherwise("complete"))
        .withColumn("order_date", F.date_add(F.lit("2026-06-01"), (F.col("id") % 60).cast("int")))
        .repartition(24)  # deliberately many small files so the audit has something to find
        .write.format("delta").mode("overwrite").save(f"{TABLES_ROOT}/orders"))
    spark.sql(f"""CREATE TABLE IF NOT EXISTS delta.`{TABLES_ROOT}/orders_silver`
        (order_id BIGINT, customer_id INT, amount DOUBLE, status STRING, order_date DATE)
        USING DELTA TBLPROPERTIES('delta.enableDeletionVectors'='true','delta.enableChangeDataFeed'='true')""")
    src = spark.read.format("delta").load(f"{TABLES_ROOT}/orders").withColumnRenamed("id","order_id")
    src.write.format("delta").mode("append").save(f"{TABLES_ROOT}/orders_silver")
    spark.sql(f"DELETE FROM delta.`{TABLES_ROOT}/orders_silver` WHERE status='cancelled'")
    print("demo tables created: orders (24 small files), orders_silver (DV+CDF, has deletes)")
else:
    print("existing tables found - demo data skipped")

existing tables found - demo data skipped


In [4]:
def discover_delta_tables(root):
    """Platform-first discovery: a Delta table is a directory with a _delta_log, full stop."""
    found = []
    for d in sorted(os.listdir(root)):
        p = os.path.join(root, d)
        if os.path.isdir(os.path.join(p, "_delta_log")) and not d.startswith("_"):
            found.append((d, p))
    return found

def audit_table(name, path):
    detail = spark.sql(f"DESCRIBE DETAIL delta.`{path}`").collect()[0]
    props  = {r["key"]: r["value"] for r in spark.sql(f"SHOW TBLPROPERTIES delta.`{path}`").collect()}
    hist   = spark.sql(f"DESCRIBE HISTORY delta.`{path}`").select(
                 "version","timestamp","operation","operationMetrics").collect()
    last_op = {}
    dv_rows_pending = 0
    for h in hist:
        if h["operation"] not in last_op:
            last_op[h["operation"]] = h["timestamp"]
        m = h["operationMetrics"] or {}
        dv_rows_pending += int(m.get("numDeletionVectorsAdded", 0) or 0)
        if h["operation"] in ("OPTIMIZE", "REORG"):
            break  # DV counting only since last compaction
    n_files = detail["numFiles"] or 0
    size_mb = (detail["sizeInBytes"] or 0) / 1024 / 1024
    avg_mb  = size_mb / n_files if n_files else 0.0
    flags = []
    if n_files >= MIN_FILES and avg_mb < SMALL_FILE_MB: flags.append("COMPACT")
    if props.get("delta.enableDeletionVectors") == "true" and dv_rows_pending > 0: flags.append("PURGE_DV")
    if "OPTIMIZE" not in last_op and size_mb > 256: flags.append("NEVER_OPTIMIZED")
    if "VACUUM END" not in last_op and "VACUUM" not in str(last_op) and len(hist) > 20: flags.append("VACUUM_OVERDUE")
    return {"run_id": RUN_ID, "table_name": name, "path": path,
            "size_mb": round(size_mb, 1), "num_files": n_files, "avg_file_mb": round(avg_mb, 1),
            "dv_enabled": props.get("delta.enableDeletionVectors") == "true",
            "cdf_enabled": props.get("delta.enableChangeDataFeed") == "true",
            "dv_pending": dv_rows_pending,
            "last_optimize": str(last_op.get("OPTIMIZE", "never")),
            "history_len": len(hist), "flags": ",".join(flags) or "OK"}

rows = [audit_table(n, p) for n, p in discover_delta_tables(TABLES_ROOT)]
report = spark.createDataFrame(rows)
report.select("table_name","size_mb","num_files","avg_file_mb","dv_enabled","cdf_enabled","dv_pending","flags").show(truncate=False)

+-----------------+-------+---------+-----------+----------+-----------+----------+-----+
|table_name       |size_mb|num_files|avg_file_mb|dv_enabled|cdf_enabled|dv_pending|flags|
+-----------------+-------+---------+-----------+----------+-----------+----------+-----+
|orders           |1.1    |1        |1.1        |false     |false      |0         |OK   |
|orders_quarantine|0.3    |2        |0.2        |false     |false      |0         |OK   |
|orders_silver    |0.9    |1        |0.9        |true      |true       |0         |OK   |
|silver_orders_inc|1.1    |1        |1.1        |true      |true       |0         |OK   |
+-----------------+-------+---------+-----------+----------+-----------+----------+-----+



In [5]:
# Versioned report: append with run_id -> full audit history queryable, latest via max(run_id).
report.write.format("delta").mode("append").option("mergeSchema","true").save(REPORT_TABLE)
latest = (spark.read.format("delta").load(REPORT_TABLE)
          .where(F.col("run_id") == RUN_ID))
needs_action = latest.where(F.col("flags") != "OK").count()
print(f"health report v{RUN_ID}: {latest.count()} tables audited, {needs_action} flagged")
assert latest.count() > 0
spark.stop() if "local" in spark.sparkContext.master else None

health report v20260804T120729Z: 4 tables audited, 0 flagged
